# N03 — k-NN vs Decision Tree

**Pendamping S07 — Kecerdasan Buatan & ML Dasar**

Pola kerja: **DO → OBSERVE → INTERPRET → EXPLAIN**. Jalankan sel secara berurutan, catat evidence, lalu tuliskan interpretasi Anda.

## k-NN vs Decision Tree pada Iris

In [2]:
# ===== Import library =====
import pandas as pd                                    # untuk tabel hasil
from sklearn.datasets import load_iris                 # dataset Iris
from sklearn.model_selection import train_test_split   # membagi data train/test
from sklearn.pipeline import Pipeline                  # merangkai scaler + model
from sklearn.preprocessing import StandardScaler       # standarisasi fitur (mean 0, std 1)
from sklearn.neighbors import KNeighborsClassifier     # k-NN (berbasis jarak)
from sklearn.tree import DecisionTreeClassifier        # Decision Tree (berbasis split)
from sklearn.metrics import accuracy_score             # (tidak dipakai; skor dihitung lewat model.score)

# ===== Muat data & split =====
data=load_iris()                                       # 150 sampel, 4 fitur, 3 kelas
# 70% train / 30% test. stratify menjaga proporsi kelas, random_state=42 agar hasil selalu sama
X_train,X_test,y_train,y_test=train_test_split(
    data.data,data.target,test_size=.3,random_state=42,stratify=data.target
)

rows=[]                                                # penampung hasil tiap model

# ===== k-NN: coba k = 1, 5, 15 =====
for k in [1,5,15]:
    # scaler ada DI DALAM Pipeline, jadi hanya belajar dari data train (tidak bocor ke test)
    model=Pipeline([('scaler',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=k))])
    model.fit(X_train,y_train)                         # latih scaler + k-NN
    # catat akurasi train dan test
    rows.append({'model':f'kNN k={k}','train':model.score(X_train,y_train),'test':model.score(X_test,y_test)})

# ===== Decision Tree: coba max_depth = 2, 4, None (tanpa batas) =====
for d in [2,4,None]:
    model=DecisionTreeClassifier(max_depth=d,random_state=42)  # random_state agar tree konsisten
    model.fit(X_train,y_train)                         # latih tree (tanpa scaler)
    rows.append({'model':f'Tree depth={d}','train':model.score(X_train,y_train),'test':model.score(X_test,y_test)})

pd.DataFrame(rows)                                     # tampilkan tabel hasil

,model,train,test
0,kNN k=1,1.000000,0.933333
1,kNN k=5,0.980952,0.911111
2,kNN k=15,0.980952,0.933333
3,Tree depth=2,0.971429,0.888889
4,Tree depth=4,0.990476,0.888889
5,Tree depth=None,1.000000,0.933333


In [3]:
# ===== Task 1: Baseline =====
from sklearn.dummy import DummyClassifier
import numpy as np

print('Jumlah train/test      :', len(X_train), '/', len(X_test))
print('Sampel per kelas train :', np.bincount(y_train))   # harus seimbang (stratified)
print('Sampel per kelas test  :', np.bincount(y_test))

# Baseline paling sederhana: selalu menebak kelas mayoritas (tidak belajar pola apa pun)
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('Baseline train/test    :', round(dummy.score(X_train, y_train), 4), '/', round(dummy.score(X_test, y_test), 4))

# Tabel hasil + kolom gap (train - test) untuk melihat overfitting
hasil = pd.DataFrame(rows)
hasil['gap'] = hasil['train'] - hasil['test']
hasil.round(4)

Jumlah train/test      : 105 / 45
,Sampel per kelas train : [35 35 35]
,Sampel per kelas test  : [15 15 15]
,Baseline train/test    : 0.3333 / 0.3333


,model,train,test,gap
0,kNN k=1,1.0000,0.9333,0.0667
1,kNN k=5,0.9810,0.9111,0.0698
2,kNN k=15,0.9810,0.9333,0.0476
3,Tree depth=2,0.9714,0.8889,0.0825
4,Tree depth=4,0.9905,0.8889,0.1016
5,Tree depth=None,1.0000,0.9333,0.0667


In [4]:
# Tabel train vs accuracy per model
import matplotlib.pyplot as plt  # dipakai untuk visualisasi pohon pada bagian interpretability

tabel_train_accuracy = hasil[['model', 'train', 'test']].rename(
    columns={'model': 'Model', 'train': 'Train', 'test': 'Accuracy'}
)
tabel_train_accuracy.round(4)

Model,Train,Accuracy
kNN k=1,1.0000,0.9333
kNN k=5,0.9810,0.9111
kNN k=15,0.9810,0.9333
Tree depth=2,0.9714,0.8889
Tree depth=4,0.9905,0.8889
Tree depth=None,1.0000,0.9333


### Catatan eksperimen Task 1–3

**Task 1 — Baseline**
- **OBSERVE:** split stratified menghasilkan 105 train dan 45 test, masing-masing kelas 35 dan 15 sampel. Baseline kelas mayoritas hanya mendapat 0.333 (train dan test).
- **INTERPRET:** karena tiga kelas seimbang, menebak kelas yang sama selalu setara peluang 1/3. Semua model (0.889–0.933 di test) jauh di atas itu, artinya keempat fitur Iris memang informatif.
- **EXPLAIN:** stratify menjaga test set seimbang. Limitation: test hanya 45 sampel, jadi 1 sampel salah = 2.2 poin persen akurasi.

**Task 2 — k-NN (scaler dalam Pipeline)**
- **OBSERVE:** k=1 train 1.000 / test 0.933; k=5 train 0.981 / test 0.911; k=15 train 0.981 / test 0.933.
- **INTERPRET:** k=1 hanya melihat satu tetangga (tetangga terdekat titik train adalah dirinya sendiri), sehingga train sempurna dan batas keputusan paling rumit (variance tinggi). k besar merata-ratakan lebih banyak tetangga, batas lebih halus (bias naik).
- **EXPLAIN:** perbedaan test antar k hanya 1 sampel, jadi belum cukup untuk menyebut satu k lebih baik dari split ini saja. Limitation: hanya 3 nilai k dan 1 kali split.

**Task 3 — Decision Tree (max_depth)**
- **OBSERVE:** depth=2 train 0.971 / test 0.889; depth=4 train 0.990 / test 0.889; depth=None train 1.000 / test 0.933.
- **INTERPRET:** makin dalam tree, makin banyak aturan sehingga train naik (kapasitas naik), tetapi test tidak ikut naik secara konsisten. Tree tanpa batas menghafal data train (100%).
- **EXPLAIN:** train accuracy 100% bukan bukti tree bagus. Test 0.933 pada depth=None kemungkinan kebetulan satu split (selisih 2 sampel), sehingga hasil satu split ini perlu dibaca dengan hati-hati. Limitation: hanya 3 nilai depth.

### Tugas
1. Model mana yang memiliki gap train–test besar?
2. Apa efek k kecil/besar?
3. Apa efek depth?
4. Mengapa k-NN memakai scaling sementara tree umumnya tidak wajib?

### Jawaban Tugas

**1. Model dengan gap train–test besar.**
- Gap terbesar terdapat pada Tree depth=4 (0,102), sedangkan terkecil pada kNN k=15 (0,048).
- kNN k=1 dan Tree depth=None paling berpotensi mengalami overfitting karena akurasi train mencapai 100%.
- Perbedaan akurasi antar model hanya 1–2 sampel, sehingga hasil perlu ditafsirkan dengan hati-hati.
  
**2. Efek k kecil dan k besar.**
- k=1 lebih sensitif terhadap noise dan berisiko overfitting karena train mencapai 100%.
- k=15 lebih stabil, tetapi berisiko underfitting jika terlalu besar.
- Akurasi test antar nilai k hampir sama, sehingga belum ada yang terbukti lebih unggul.

**3. Efek depth.**
- Semakin besar depth, train accuracy meningkat hingga 100%, tetapi test accuracy tidak selalu naik.
- Tree yang lebih dalam cenderung menghafal data train (overfitting).
- Tree depth=2 sudah mampu memisahkan ketiga kelas berdasarkan petal length dan petal width.
  
**4. Mengapa k-NN memakai scaling dan tree umumnya tidak.**
- k-NN menggunakan perhitungan jarak, sehingga membutuhkan scaling agar semua fitur seimbang.
- Decision Tree tidak membutuhkan scaling karena membandingkan fitur berdasarkan nilai threshold.
- Pada Iris, pengaruh scaling kecil karena semua fitur memiliki skala yang mirip.

### Interpretability: aturan Decision Tree vs k-NN

In [ ]:
from sklearn.tree import export_text, plot_tree

# Tree sederhana (depth=2) dilatih ulang untuk dibaca sebagai aturan if-then
tree2 = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
print(export_text(tree2, feature_names=list(data.feature_names)))   # aturan dalam bentuk teks

# Kompleksitas tree pada tiap depth
for d in [2, 4, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    print(f'max_depth={d}: depth aktual={t.get_depth()}, jumlah daun={t.get_n_leaves()}')

# Gambar tree depth=2
plt.figure(figsize=(9, 4.5))
plot_tree(tree2, feature_names=data.feature_names, class_names=list(data.target_names), filled=True)
plt.show()

### Task 4 — Tabel perbandingan

In [10]:
# Representatif: k=5 (default sklearn), depth=2 (paling sederhana), depth=None (default sklearn)
hs = hasil.set_index('model')
def baris(nama, pre, kompleks, interp):
    return {'Model': nama,
            'Test (1 split)': round(hs.loc[nama, 'test'], 3),
            'CV mean ± std': f"{cv_hasil.loc[nama, 'cv_mean']:.3f} ± {cv_hasil.loc[nama, 'cv_std']:.3f}",
            'Preprocessing': pre, 'Complexity': kompleks, 'Interpretability': interp}

pd.set_option('display.max_colwidth', None)   # tampilkan teks sel secara penuh
tabel_task4 = pd.DataFrame([
    baris('kNN k=5',
          'Wajib scaling (berbasis jarak)',
          'Latih hampir tanpa biaya, tetapi menyimpan seluruh data train; prediksi menghitung jarak ke semua titik train',
          'Rendah: tidak ada aturan global, penjelasan hanya berupa tetangga terdekat'),
    baris('Tree depth=2',
          'Tidak wajib (split per fitur)',
          'Sangat kecil: 3 daun, prediksi hanya menelusuri 2 pertanyaan',
          'Tinggi: aturan if-then yang bisa dibaca'),
    baris('Tree depth=None',
          'Tidak wajib (split per fitur)',
          'Lebih besar: depth 5, 8 daun; mudah menghafal train',
          'Sedang: masih terbaca, tetapi aturan makin banyak dan spesifik'),
])
tabel_task4

,Model,Test (1 split),CV mean ± std,Preprocessing,Complexity,Interpretability
0,kNN k=5,0.911,0.933 ± 0.049,Wajib scaling (berbasis jarak),"Latih hampir tanpa biaya, tetapi menyimpan seluruh data train; prediksi menghitung jarak ke semua titik train","Rendah: tidak ada aturan global, penjelasan hanya berupa tetangga terdekat"
1,Tree depth=2,0.889,0.943 ± 0.036,Tidak wajib (split per fitur),"Sangat kecil: 3 daun, prediksi hanya menelusuri 2 pertanyaan",Tinggi: aturan if-then yang bisa dibaca
2,Tree depth=None,0.933,0.933 ± 0.023,Tidak wajib (split per fitur),"Lebih besar: depth 5, 8 daun; mudah menghafal train","Sedang: masih terbaca, tetapi aturan makin banyak dan spesifik"


**Task 4 — Compare**
- **OBSERVE:** skor ketiga model hampir sama (test 0.889–0.933, CV 0.933–0.943), tetapi preprocessing, ukuran model, dan kemudahan penjelasan berbeda jelas.
- **INTERPRET:** k-NN *lazy/instance-based* (menyimpan data, hitung jarak saat prediksi) sehingga sensitif skala dan sulit dijelaskan secara global. Tree mempelajari aturan eksplisit sehingga ringkas dan terbaca, tetapi depth yang tidak dibatasi menambah risiko overfit.
- **EXPLAIN:** karena skor setara, pemilihan model sebaiknya berdasarkan kebutuhan: tree depth kecil bila butuh penjelasan dan preprocessing minimal, k-NN bila fitur berskala sama atau sudah di-scale. Limitation: kesimpulan ini hanya pada Iris yang kecil dan bersih.

### Evidence Log

| Eksperimen | Perubahan | Metric / Output | Interpretasi |
|---|---|---|---|
| EXP-01 | Baseline kelas mayoritas vs model, split stratified 70/30 | Baseline 0.333; semua model test 0.889–0.933 | Fitur informatif, model jauh di atas tebakan. Test hanya 45 sampel (1 sampel = 2.2 poin) |
| EXP-02 | k-NN k = 1, 5, 15 (scaler dalam Pipeline) | train 1.000 / 0.981 / 0.981; test 0.933 / 0.911 / 0.933 | k=1 menghafal train. Selisih test hanya 1 sampel, belum bisa disimpulkan |
| EXP-03 | Tree max_depth = 2, 4, None | train 0.971 / 0.990 / 1.000; test 0.889 / 0.889 / 0.933; gap 0.083 / 0.102 / 0.067. Tabel Train dan Accuracy ditampilkan di atas | Train naik dengan depth, test tidak konsisten. Train 100% bukan bukti tree bagus |

## Refleksi akhir

1. Apa hasil utama yang Anda amati?
2. Mengapa hasil tersebut terjadi menurut konsep di Main Deck?
3. Apa limitation eksperimen ini?
4. Eksperimen apa yang layak dilakukan berikutnya?

### Jawaban Refleksi

**1. Hasil utama.**
- Semua model memiliki akurasi test tinggi, yaitu 0,889–0,933.
- Semakin kompleks model, akurasi train meningkat, tetapi test tidak selalu meningkat.
- Perbedaan hasil antar model sangat kecil, sehingga belum ada model yang paling unggul.
- Decision Tree tanpa batas memiliki akurasi test tertinggi, yaitu 0,933, meskipun train mencapai 100%.
  
**2. Penjelasan dengan konsep.**
- Model kompleks cenderung overfitting karena menghafal data train, sedangkan model sederhana lebih stabil.
- k-NN membutuhkan scaling karena menggunakan perhitungan jarak.
- Decision Tree tidak terpengaruh skala fitur dan aturannya mudah dipahami.

**3. Limitation.**
- Data test hanya 45 sampel, sehingga hasil bisa berubah tergantung pembagian data.
- Hyperparameter yang diuji masih terbatas.
- Evaluasi hanya menggunakan accuracy, tanpa metrik lainnya.
- Dataset Iris masih sederhana, sehingga hasil belum tentu berlaku pada data yang lebih kompleks.

**4. Eksperimen berikutnya.**
- Menguji lebih banyak nilai k dan depth menggunakan Cross Validation.
- Menggunakan GridSearchCV untuk mencari parameter terbaik.
- Menambahkan confusion matrix untuk melihat kesalahan prediksi.
- Mencoba dataset lain seperti Wine atau Breast Cancer.
- Menguji metode jarak k-NN dan pruning pada Decision Tree.